# Google Trends in Colab without 429 errors

pytrends calls Google's unofficial endpoints from your own IP. Colab notebooks share Google Cloud IPs, so pytrends usually fails here with `TooManyRequestsError: The request failed: Google returned a response with code 429`.

[pytrends-cloud](https://github.com/retracn/pytrends-cloud) keeps the pytrends interface, so the code below is ordinary pytrends code. The only change is the import. Each query runs on a hosted Google Trends API that rotates IPs and retries for you.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/retracn/pytrends-cloud

## 2. Add your Apify token

pytrends-cloud fetches through a hosted Google Trends API on Apify. Create a [free Apify account](https://console.apify.com/sign-up) (the free plan's $5 monthly credit covers about 5,000 keyword reports) and copy your token from [Settings → API & Integrations](https://console.apify.com/settings/integrations).

In Colab, click the key icon in the left sidebar, add a secret named `APIFY_TOKEN` and turn on notebook access. The cell below reads it, or asks for the token if you skip that.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["APIFY_TOKEN"] = userdata.get("APIFY_TOKEN")
except Exception:
    pass
if not os.environ.get("APIFY_TOKEN"):
    os.environ["APIFY_TOKEN"] = getpass("Apify API token: ")

## 3. Interest over time

The same calls as pytrends. One `build_payload` query is fetched once: interest over time, regions and related queries all come from that one request.

In [ ]:
from pytrends_cloud.request import TrendReq   # instead of: from pytrends.request import TrendReq

pytrends = TrendReq(hl="en-US", tz=0)
pytrends.build_payload(["chatgpt", "claude", "gemini"], timeframe="today 12-m", geo="US")

df = pytrends.interest_over_time()
df.tail()

In [ ]:
df.drop(columns="isPartial").plot(figsize=(10, 4), title="Google search interest, US, past 12 months");

## 4. Where people search most, and what else they search

In [ ]:
regions = pytrends.interest_by_region(resolution="REGION")
regions.sort_values("claude", ascending=False).head(10)

In [ ]:
related = pytrends.related_queries()
related["claude"]["rising"]

## 5. What's trending right now

In [ ]:
pytrends.trending_searches(pn="united_kingdom").head(20)

## Cost and limits

- $1 per 1,000 keyword reports on your Apify account; a query is one report per keyword, and a region breakdown at another resolution is a second query, so this notebook costs under a cent. Trending searches cost $0.50 per 1,000.
- `related_topics()` returns empty results: Google withholds related topics from automated requests.
- Other timeframes work as in pytrends: `now 7-d`, `today 5-y`, `all`, or `"2024-01-01 2024-12-31"`.

More: [pytrends-cloud on GitHub](https://github.com/retracn/pytrends-cloud) · [the Google Trends Scraper it runs on](https://apify.com/automationnation/google-trends-scraper)